# Análise das exportações de Plantas Vivas e Produtos de Floricultura no RS por Porto/URF (2015–2025)

Notebook adaptado no mesmo formato das análises anteriores, usando o arquivo `data_RS_Plantas-Floricultura.xlsx`.

Gráficos incluídos:
1. Evolução do peso exportado por Porto/URF
2. Evolução dos 3 maiores Portos/URFs
3. Participação percentual por URF no total exportado de Plantas Vivas e Produtos de Floricultura


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from IPython.display import display

arquivo = "data_RS_Plantas-Floricultura.xlsx"
abas = pd.read_excel(arquivo, sheet_name=None)

nomes_portos = {'1010252': '1010252 - Jaguarão', '1010253': '1010253 - Bagé', '1010351': '1010351 - IRF Santana do Livramento', '1010900': '1010900 - Uruguaiana', '1010953': '1010953 - São Borja', '1015500': '1015500 - Chuí', '1015600': '1015600 - Santana do Livramento', '1017500': '1017500 - ALF Uruguaiana', '1017503': '1017503 - IRF São Borja', '1017600': '1017600 - Aeroporto Salgado Filho', '1017700': '1017700 - Porto de Rio Grande', '1017701': '1017701 - IRF Chuí', '1017800': '1017800 - ALF Porto Alegre', '1017801': '1017801 - IRF Aeroporto Salgado Filho', '1017900': '1017900 - ALF Santana do Livramento'}

lista_dfs = []

for nome_aba, df in abas.items():
    df = df.copy()

    codigo_match = re.search(r"\d{7}", nome_aba)
    if codigo_match:
        codigo = codigo_match.group()
        df["Porto_URF"] = nomes_portos.get(codigo, f"{codigo} - Nome não mapeado")
        lista_dfs.append(df)

base = pd.concat(lista_dfs, ignore_index=True)

base["Peso_KG"] = pd.to_numeric(base["Peso_KG"], errors="coerce").fillna(0)
base["Valor_USD"] = pd.to_numeric(base["Valor_USD"], errors="coerce").fillna(0)
base["Ano"] = pd.to_numeric(base["Ano"], errors="coerce")

base.head()


In [ ]:
df_porto_ano = (
    base
    .groupby(["Ano", "Porto_URF"], as_index=False)["Peso_KG"]
    .sum()
)

df_porto_ano["Peso_milhoes"] = df_porto_ano["Peso_KG"] / 1_000_000

total_por_porto = (
    df_porto_ano
    .groupby("Porto_URF", as_index=False)["Peso_KG"]
    .sum()
    .sort_values("Peso_KG", ascending=False)
)

total_por_porto["Participacao_%"] = total_por_porto["Peso_KG"] / total_por_porto["Peso_KG"].sum() * 100

display(total_por_porto)


## 1. Evolução do peso exportado de Plantas Vivas e Produtos de Floricultura por Porto/URF

In [ ]:
grafico = df_porto_ano.copy()

tabela_grafico = grafico.pivot(
    index="Ano",
    columns="Porto_URF",
    values="Peso_milhoes"
).fillna(0)

plt.figure(figsize=(18, 8))

for porto in tabela_grafico.columns:
    plt.plot(
        tabela_grafico.index,
        tabela_grafico[porto],
        marker="o",
        label=porto
    )

plt.title("Evolução do Peso Exportado de Plantas Vivas e Produtos de Floricultura por Porto/URF")
plt.xlabel("Ano")
plt.ylabel("Peso exportado de plantas vivas e produtos de floricultura (milhões de KG)")
plt.xticks(tabela_grafico.index)

plt.legend(
    title="Porto/URF",
    bbox_to_anchor=(0.5, -0.18),
    loc="upper center",
    ncol=2,
    fontsize=8
)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 2. Evolução dos 3 maiores Portos/URFs

In [ ]:
grafico = df_porto_ano.copy()

# Identificar os 3 maiores exportadores no total do período
top3_portos = (
    grafico
    .groupby("Porto_URF")["Peso_milhoes"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index
)

# Filtrar apenas o Top 3
grafico_top3 = grafico[grafico["Porto_URF"].isin(top3_portos)]

# Criar tabela para o gráfico
tabela_grafico = grafico_top3.pivot_table(
    index="Ano",
    columns="Porto_URF",
    values="Peso_milhoes",
    aggfunc="sum",
    fill_value=0
)

plt.figure(figsize=(16, 7))

for porto in tabela_grafico.columns:
    plt.plot(
        tabela_grafico.index,
        tabela_grafico[porto],
        marker="o",
        linewidth=2,
        label=porto
    )

plt.title("Evolução do Peso Exportado de Plantas Vivas e Produtos de Floricultura pelos 3 Maiores Portos/URFs")
plt.xlabel("Ano")
plt.ylabel("Peso exportado de plantas vivas e produtos de floricultura (milhões de KG)")
plt.xticks(tabela_grafico.index)

plt.legend(
    title="Porto/URF",
    bbox_to_anchor=(0.5, -0.18),
    loc="upper center",
    ncol=2,
    fontsize=9
)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Participação percentual por URF no total exportado

In [ ]:
import matplotlib.patheffects as path_effects

nomes_portos_legenda = {'1010252': '1010252 - JAGUARAO', '1010253': '1010253 - BAGE', '1010351': '1010351 - IRF SANTANA DO LIVRAMENTO', '1010900': '1010900 - URUGUAIANA', '1010953': '1010953 - SAO BORJA', '1015500': '1015500 - CHUI', '1015600': '1015600 - SANTANA DO LIVRAMENTO', '1017500': '1017500 - ALF - URUGUAIANA', '1017503': '1017503 - IRF - SÃO BORJA', '1017600': '1017600 - AEROPORTO SALGADO FILHO - PORTO ALEGRE', '1017700': '1017700 - PORTO DE RIO GRANDE', '1017701': '1017701 - IRF - CHUÍ', '1017800': '1017800 - ALF - PORTO ALEGRE', '1017801': '1017801 - IRF - AEROPORTO INTERNACIONAL SALGADO FILHO', '1017900': '1017900 - ALF - SANTANA DO LIVRAMENTO'}

lista_dfs_pizza = []
for nome_aba, df in abas.items():
    codigo_match = re.search(r"\d{7}", nome_aba)
    if codigo_match:
        codigo = codigo_match.group()
        df = df.copy()
        df["Porto_URF"] = nomes_portos_legenda.get(codigo, f"{codigo} - Nome não mapeado")
        lista_dfs_pizza.append(df)

base_pizza = pd.concat(lista_dfs_pizza, ignore_index=True)
base_pizza["Peso_KG"] = pd.to_numeric(base_pizza["Peso_KG"], errors="coerce").fillna(0)

participacao = (
    base_pizza
    .groupby("Porto_URF", as_index=False)["Peso_KG"]
    .sum()
    .sort_values("Peso_KG", ascending=False)
)
participacao["Percentual"] = participacao["Peso_KG"] / participacao["Peso_KG"].sum() * 100

labels = [f"{row.Porto_URF} ({row.Percentual:.2f}%)" for row in participacao.itertuples()]
values = participacao["Peso_KG"]

fig = plt.figure(figsize=(16, 9), facecolor="#f2f2f2")
ax = fig.add_axes([0.03, 0.06, 0.55, 0.84], facecolor="#f2f2f2")

def autopct_func(pct):
    return f"{pct:.1f}%" if pct >= 3 else ""

wedges, texts, autotexts = ax.pie(
    values,
    startangle=90,
    counterclock=True,
    autopct=autopct_func,
    pctdistance=0.68,
    wedgeprops={"linewidth": 0.8, "edgecolor": "white"},
    textprops={"color": "white", "fontsize": 20, "weight": "bold"},
)

for txt in autotexts:
    txt.set_path_effects([
        path_effects.Stroke(linewidth=3, foreground="black", alpha=0.35),
        path_effects.Normal()
    ])

ax.axis("equal")

fig.suptitle(
    "Participação percentual por URF no total exportado de Plantas Vivas e Produtos de Floricultura (2015–2025)",
    fontsize=22,
    fontweight="bold",
    y=0.96,
)

legend = fig.legend(
    wedges,
    labels,
    title="URFs",
    loc="center left",
    bbox_to_anchor=(0.60, 0.50),
    frameon=True,
    fontsize=11,
    title_fontsize=16,
)
legend.get_title().set_weight("bold")
legend.get_frame().set_facecolor("white")
legend.get_frame().set_edgecolor("#999999")
legend.get_frame().set_linewidth(0.8)

plt.savefig("participacao_percentual_urf_plantas_floricultura.png", dpi=150, facecolor=fig.get_facecolor(), bbox_inches="tight")
plt.show()
